In [ ]:
import warnings
import os
import cupy as cp
import pandas as pd
import cudf
import dask_cudf
import pandas as pd
from io import StringIO
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
df = cudf.read_csv('DaneModelTMP.csv', delimiter=',')

In [ ]:
import numpy as np
import warnings
import cuml
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from cuml.naive_bayes import GaussianNB
from cuml.cluster import DBSCAN
from cuml.neighbors import KNeighborsClassifier
from sklearn import model_selection
from cuml.model_selection import train_test_split
from cuml.linear_model import Lasso
from cuml.common.device_selection import using_device_type, set_global_device_type
# cuml.set_global_output_type('cudf')

# Create instances of the estimators
rf = RandomForestClassifier()
dtree = DecisionTreeClassifier()
gnb = GaussianNB()
dbscan = DBSCAN()  # Assuming you want to use DBSCAN as an estimator, which is unusual
knc = KNeighborsClassifier(n_neighbors=3)
la = Lasso()

# List of (name, estimator) tuples
def EksperymentMLRegresja(X,y, out_dtype='float64'):
    dfs=[]
    wyniki=[]
    lista_estymatorow = [
        ('rfc', rf),
        ('Dtreec', dtree),
        ('gnb', gnb),
        ('KNC', knc),
        ('la', la)
    ]

# Create the StackingClassifier
    miary=['accuracy','r2']
    for nazwa, model in lista_estymatorow:
        podzial=model_selection.KFold(n_splits=5,shuffle=True,random_state=125)
        with using_device_type('gpu'):
            history = model.fit(X,y)
            cv_wynik=model_selection.cross_validate(model,X,y,cv=podzial,scoring=miary)
            wyniki.append(cv_wynik)
            df=pd.DataFrame(cv_wynik)
            df['model']= nazwa
            dfs.append(df)
    # model = stack_clf = StackingClassifier(estimators=lista_estymatorow)
    final=pd.concat(dfs)
    return final

X, y = df[['Pora', 'Zachm3', \
'Zachm6', 'ZachmN3', 'ZachmN6', 'Wiatr_N', 'Wiatr_E', 'Wiatr_S', \
'Wiatr_W', 'DeltaTemp3', 'TemperaturaPowietrza', 'DeltaTemp6', \
'DeltaTempRosy3', 'DeltaTempRosy6', 'TemperaturaPunktuRosy', \
'Cisn3', 'Cisn6', 'CisnienieNaPoziomieStacji', 'Opady6', 'Opady12', \
'Mgla6', 'Mzawka6', 'Deszcz6', 'Snieg6', 'Przelotny6', 'Burza6', \
'Mgla_W', 'Mgla_E', 'Mgla_N', 'Mgla_S', 'Mzawka_W', 'Mzawka_E', \
'Mzawka_N', 'Mzawka_S', 'Burza_W', 'Burza_E', 'Burza_N', 'Burza_S', \
'Deszcz_W', 'Deszcz_E', 'Deszcz_N', 'Deszcz_S', 'Snieg_W', \
'Snieg_E', 'Snieg_N', 'Snieg_S', 'OpadM_W', 'OpadM_E', 'OpadM_N', \
'OpadM_S', 'Przelotny_W', 'Przelotny_E', 'Przelotny_N', \
'Przelotny_S']], df.Pred6

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

f_X_train = X_train.to_numpy()
f_y_train = y_train.to_numpy()

f_X_train = f_X_train.astype(np.float64)
f_y_train = f_y_train.astype(np.float64)

wynik_koncowy = EksperymentMLRegresja(f_X_train, f_y_train, out_dtype='float64')
print(wynik_koncowy)